In [ ]:
# ── COLAB SETUP ────────────────────────────────────────────────────────────────
# Run this cell first. It installs AaronTools and embeds the required Gaussian
# output file (water.log) directly — no external downloads needed.
#
# numpy and matplotlib are pre-installed in Google Colab.
# AaronTools will be installed below (takes ~30 seconds on first run).

import sys, os, subprocess, platform
print(f"Python {sys.version}")
print(f"Platform: {platform.system()}")

# ── Install AaronTools ────────────────────────────────────────────────────────
print("\nInstalling AaronTools...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "AaronTools"])
print("AaronTools installed ✓")

# ── Embed water.log ───────────────────────────────────────────────────────────
# B3LYP/6-31G* geometry optimization and harmonic frequency calculation for H2O
_WATER_LOG_CONTENT = """
 Entering Gaussian System, Link 0=g16
 %chk=water.chk
 %mem=4GB
 %nprocshared=4
 ----------------------------------------------------------------------
 #p B3LYP/6-31G* Opt Freq

 Water molecule optimization and frequency calculation

 ----------------------------------------------------------------------
 Symbolic Z-matrix:
 Charge =  0 Multiplicity = 1
 O
 H  1  r1
 H  1  r1  2  a1

       Variables:
  r1                    0.96
  a1                  104.5


                          Input orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.000000
      2          1           0        0.000000    0.000000    0.960000
      3          1           0        0.927516    0.000000   -0.240000
 ---------------------------------------------------------------------

 SCF Done:  E(RB3LYP) =  -76.4083579211     A.U. after    9 cycles
            NFock= 9  Conv=0.47D-08     -V/T= 2.0089

 SCF Done:  E(RB3LYP) =  -76.4186251034     A.U. after   10 cycles
            NFock=10  Conv=0.38D-08     -V/T= 2.0091

 SCF Done:  E(RB3LYP) =  -76.4187385461     A.U. after    9 cycles
            NFock= 9  Conv=0.23D-08     -V/T= 2.0091

                         Standard orientation:
 ---------------------------------------------------------------------
 Center     Atomic      Atomic             Coordinates (Angstroms)
 Number     Number       Type             X           Y           Z
 ---------------------------------------------------------------------
      1          8           0        0.000000    0.000000    0.117176
      2          1           0        0.000000    0.757306   -0.468706
      3          1           0        0.000000   -0.757306   -0.468706
 ---------------------------------------------------------------------

 Rotational constants (GHZ):         836.3077963         434.9876987         286.8513882

 Zero-point correction=                           0.020908 (Hartree/Particle)
 Thermal correction to Energy=                    0.023742
 Thermal correction to Enthalpy=                  0.024686
 Thermal correction to Gibbs Free Energy=         0.003610
 Sum of electronic and zero-point Energies=     -76.397830
 Sum of electronic and thermal Energies=        -76.394996
 Sum of electronic and thermal Enthalpies=      -76.394052
 Sum of electronic and thermal Free Energies=  -76.415128

 E (Thermal)             CV                CP                S
 KCal/Mol        Cal/Mol-Kelvin    Cal/Mol-Kelvin    Cal/Mol-Kelvin
 14.896              6.002             8.314             44.278

 Zero-point vibrational energy     130943.2 (Joules/Mol)
                                    31.298 (Kcal/Mol)

 Frequencies --   1589.0765                3703.9127                3807.0960
 Red. masses --      1.0831                   1.0451                   1.0818
 Frc consts  --      1.6145                   8.4518                   9.2448
 IR Inten    --     67.1785                  10.4654                  49.3234

 Normal termination of Gaussian 16 at Mon Mar 14 12:34:56 2026.
"""

if not os.path.exists("water.log"):
    with open("water.log", "w") as _f:
        _f.write(_WATER_LOG_CONTENT)
    print("Written : water.log")
else:
    print("Found existing: water.log")

print("\nEnvironment ready ✓")


# Python for Chemists — Part 3: Structure Alignment with AaronTools
## 🔑 Instructor Version — Solutions Included

**Prerequisites:** Parts 1 and 2. You should be comfortable with Python basics, file I/O, and parsing Gaussian output.

**What is structure alignment?**  
When Gaussian (or any QM program) optimizes a molecule, it places it somewhere arbitrary in 3-D space. Two optimized conformers of the same molecule will usually be in completely different positions and orientations even if they look nearly identical. Before you can meaningfully compare their geometries — or ask "how similar are these two structures?" — you must first remove that arbitrary translation and rotation. That process is called **structure alignment** (or **superposition**).

Alignment is the foundation of many tasks you will encounter in computational chemistry:
- Comparing an optimized geometry to an X-ray crystal structure
- Deciding whether two conformer search results are truly distinct
- Computing a reaction path RMSD to quantify how much a structure changes along an IRC
- Filtering large conformer ensembles to remove duplicates

**AaronTools** is a Python toolkit developed by the Wheeler group for automating quantum chemistry workflows. Its `Geometry` class can read structures directly from Gaussian `.log` and `.com` files (as well as `.xyz`, `.pdb`, and many others), perform RMSD-based alignment, and write results back to disk.

### Topics covered

| Section | Topic |
|---------|-------|
| 1 | Installation and imports |
| 2 | Reading molecular structures |
| 3 | Inspecting a geometry |
| 4 | Center of mass and coordinate manipulation |
| 5 | RMSD without alignment — measuring raw difference |
| 6 | RMSD with alignment — the Kabsch algorithm |
| 7 | Heavy-atom RMSD |
| 8 | Targeted alignment — aligning on a fragment |
| 9 | RMSD_permute — handling symmetric molecules |
| 10 | Filtering a conformer ensemble |
| 11 | Connecting to Gaussian: geometry from a `.log` file |

> **Instructor version:** Each **✏️ Practice** cell contains the complete solution preceded by a `# SOLUTION` comment.  
> Use `part3_student.ipynb` when distributing to students.

---
## Section 1 — Installation and Imports

AaronTools is installed with pip:

```bash
pip install AaronTools
```

The two classes you will use throughout this notebook are:
- **`Geometry`** — the central object representing a molecular structure (atoms + coordinates + connectivity)
- **`FileReader`** — a lower-level file parser; `Geometry` can accept one directly, but you will rarely need to use `FileReader` explicitly

AaronTools uses numpy arrays internally, so numpy will also be imported.

In [ ]:
from AaronTools.geometry import Geometry
from AaronTools.fileIO import FileReader
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import AaronTools
print("AaronTools version:", AaronTools.__version__)
print("All imports successful!")

---
## Section 2 — Reading Molecular Structures

AaronTools can read structures from many file formats that you will encounter in a QM lab: Gaussian `.log` output files, Gaussian `.com`/`.gjf` input files, `.xyz` files, `.pdb` files, and more.

The simplest way to load a structure is to pass a filename directly to `Geometry`:

```python
mol = Geometry("molecule.xyz")       # from XYZ file
mol = Geometry("opt.log")            # from Gaussian output
mol = Geometry("molecule.com")       # from Gaussian input
```

When reading a Gaussian `.log` file, AaronTools extracts the **last** Standard Orientation block — which corresponds to the final optimized geometry if the job converged.

The setup cell below writes the sample `.xyz` files we will use throughout this notebook — ethane in its **staggered** (ground state) and **eclipsed** (transition state) conformations. These are the two limiting geometries of the simplest C−C internal rotation.

In [ ]:
# ── Create sample XYZ files ───────────────────────────────────────────────
# Ethane staggered (H-C-C-H dihedral = 60°) — the ground state
staggered_xyz = """8
Ethane staggered conformer
C   0.000   0.000   0.765
C   0.000   0.000  -0.765
H   1.025   0.000   1.157
H  -0.512   0.887   1.157
H  -0.512  -0.887   1.157
H   0.512   0.887  -1.157
H  -1.025   0.000  -1.157
H   0.512  -0.887  -1.157
"""

# Ethane eclipsed (H-C-C-H dihedral = 0°) — the rotational transition state
eclipsed_xyz = """8
Ethane eclipsed conformer
C   0.000   0.000   0.765
C   0.000   0.000  -0.765
H   1.025   0.000   1.157
H  -0.512   0.887   1.157
H  -0.512  -0.887   1.157
H   1.025   0.000  -1.157
H  -0.512   0.887  -1.157
H  -0.512  -0.887  -1.157
"""

# Ethane randomly displaced — same staggered geometry but translated and rotated
# (as if two Gaussian jobs placed the molecule in different spots)
displaced_xyz = """8
Ethane staggered displaced
C   3.412   1.765   0.218
C   2.838   1.068  -0.987
H   4.346   1.293   0.534
H   3.587   2.791   0.015
H   2.695   1.808   1.094
H   1.900   1.538  -1.302
H   2.660   0.042  -0.790
H   3.580   1.064  -1.783
"""

with open("ethane_staggered.xyz", "w") as f:
    f.write(staggered_xyz)
with open("ethane_eclipsed.xyz", "w") as f:
    f.write(eclipsed_xyz)
with open("ethane_displaced.xyz", "w") as f:
    f.write(displaced_xyz)

print("Sample XYZ files written.")

In [ ]:
# ── EXAMPLE — load both ethane conformers ────────────────────────────────
staggered  = Geometry("ethane_staggered.xyz")
eclipsed   = Geometry("ethane_eclipsed.xyz")
displaced  = Geometry("ethane_displaced.xyz")

print(f"Staggered : {staggered.n_atoms} atoms, name = '{staggered.name}'")
print(f"Eclipsed  : {eclipsed.n_atoms} atoms")
print(f"Displaced : {displaced.n_atoms} atoms")

#### ✏️ Practice 2.1
Load `ethane_staggered.xyz` into a variable `ref` and verify:
- `n_atoms` — total number of atoms
- `n_carbons` — number of carbon atoms (hint: loop over `ref.atoms` and check `atom.element == "C"`)
- `n_hydrogens` — number of hydrogen atoms

In [ ]:
# SOLUTION — Practice 2.1
ref         = Geometry("ethane_staggered.xyz")
n_atoms     = ref.n_atoms
n_carbons   = sum(1 for a in ref.atoms if a.element == "C")
n_hydrogens = sum(1 for a in ref.atoms if a.element == "H")
print(f"✓  {n_atoms} atoms: {n_carbons} C + {n_hydrogens} H")


In [ ]:
# TEST 2.1
assert isinstance(ref, Geometry), "ref should be a Geometry object"
assert n_atoms == 8, f"Ethane has 8 atoms, got {n_atoms}"
assert n_carbons == 2, f"Ethane has 2 carbons, got {n_carbons}"
assert n_hydrogens == 6, f"Ethane has 6 hydrogens, got {n_hydrogens}"
print(f"✓  {n_atoms} atoms: {n_carbons} C + {n_hydrogens} H")

---
## Section 3 — Inspecting a Geometry Object

A `Geometry` object exposes the molecular structure through several attributes:

| Attribute | Type | Contents |
|-----------|------|----------|
| `.atoms` | `list` of `Atom` | All atom objects in order |
| `.n_atoms` | `int` | Number of atoms |
| `.coords` | `(N, 3) ndarray` | All coordinates in Å |
| `.name` | `str` | Molecule name (from comment line in XYZ, or filename) |

Each `Atom` object has:

| Attribute | Type | Contents |
|-----------|------|----------|
| `.element` | `str` | Element symbol, e.g. `"C"`, `"H"`, `"N"` |
| `.coords` | `(3,) ndarray` | xyz coordinates of this atom in Å |
| `.name` | `str` | Atom index label |

The `.coords` array on the `Geometry` object is a **copy** of the individual atom coordinates stacked into a matrix — modifying it does not move the atoms. Use `coord_shift` or `rotate` (Section 4) for that.

In [ ]:
# ── EXAMPLE — inspect the staggered ethane structure ─────────────────────
mol = Geometry("ethane_staggered.xyz")

print(f"Molecule   : {mol.name}")
print(f"Atoms      : {mol.n_atoms}")
print(f"Coord shape: {mol.coords.shape}")
print()
print("  idx  element       x          y          z")
print("  ─────────────────────────────────────────")
for i, atom in enumerate(mol.atoms):
    x, y, z = atom.coords
    print(f"  {i:3d}    {atom.element:2s}     {x:8.3f}   {y:8.3f}   {z:8.3f}")

#### ✏️ Practice 3.1
Using the `staggered` geometry object (already loaded), compute:
- `elements` — a Python list of element symbols for every atom, in order
- `formula` — a string of the molecular formula e.g. `"C2H6"` (hint: use a dictionary to count each element, then format the string)
- `c_c_dist` — the C−C bond length in Å (distance between atom 0 and atom 1, both carbons)

In [ ]:
# SOLUTION — Practice 3.1
elements = [a.element for a in staggered.atoms]

from collections import Counter
counts  = Counter(elements)
formula = "".join(f"{el}{counts[el]}" for el in sorted(counts))

c0      = staggered.atoms[0].coords
c1      = staggered.atoms[1].coords
c_c_dist = float(np.linalg.norm(c0 - c1))

print(f"Elements : {elements}")
print(f"Formula  : {formula}")
print(f"C-C dist : {c_c_dist:.4f} Å")


In [ ]:
# TEST 3.1
assert elements == [a.element for a in staggered.atoms], "elements list is incorrect"
assert elements.count("C") == 2 and elements.count("H") == 6, "element counts wrong"
assert "C2" in formula and "H6" in formula, f"Formula should be C2H6, got {formula}"
c0 = staggered.atoms[0].coords
c1 = staggered.atoms[1].coords
expected_cc = float(np.linalg.norm(c0 - c1))
assert abs(c_c_dist - expected_cc) < 1e-6, "c_c_dist is incorrect"
print(f"✓  {formula}, C−C = {c_c_dist:.3f} Å")

---
## Section 4 — Center of Mass and Coordinate Manipulation

Before computing RMSD you often need to **center** a molecule at the origin so that translation does not contribute to the comparison. AaronTools provides:

```python
com = mol.COM(mass_weight=True)   # mass-weighted center of mass (x, y, z)
com = mol.COM(mass_weight=False)  # geometric centroid (unweighted average)
```

To translate the molecule, pass a displacement vector to `coord_shift`:
```python
mol.coord_shift([dx, dy, dz])     # shifts ALL atoms by (dx, dy, dz) — modifies in place
```

To rotate the molecule around an axis passing through the origin:
```python
mol.rotate([vx, vy, vz], angle)   # angle in degrees, axis = (vx, vy, vz)
```

> **Important:** both `coord_shift` and `rotate` modify the geometry **in place**. If you want to preserve the original, use `mol.copy()` first.

In [ ]:
# ── EXAMPLE — center a molecule at the origin ────────────────────────────
mol = Geometry("ethane_staggered.xyz")

com_before = mol.COM(mass_weight=True)
print(f"COM before centering: ({com_before[0]:.6f}, {com_before[1]:.6f}, {com_before[2]:.6f})")

# Shift the molecule so the COM lands at the origin
mol.coord_shift(-com_before)

com_after = mol.COM(mass_weight=True)
print(f"COM after centering : ({com_after[0]:.2e}, {com_after[1]:.2e}, {com_after[2]:.2e})")
print("(values ~0 confirm the molecule is centred at the origin)")

#### ✏️ Practice 4.1
Working with a **copy** of `eclipsed` (so you don't modify the original):
- `mol_copy` — a copy of `eclipsed`
- Center `mol_copy` at the origin by subtracting its COM
- `com_after` — the COM of `mol_copy` after centering (should be ≈ zero)
- `max_com_component` — the largest absolute value in `com_after` (should be < 1e-10)

In [ ]:
# SOLUTION — Practice 4.1
mol_copy = eclipsed.copy()

com_before = mol_copy.COM(mass_weight=True)
mol_copy.coord_shift(-com_before)

com_after         = mol_copy.COM(mass_weight=True)
max_com_component = float(np.max(np.abs(com_after)))

print(f"COM after centering: ({com_after[0]:.2e}, {com_after[1]:.2e}, {com_after[2]:.2e})")
print(f"Max component: {max_com_component:.2e} Å")


In [ ]:
# TEST 4.1
assert mol_copy is not eclipsed, "mol_copy should be a copy, not the original object"
assert max_com_component < 1e-8, f"COM should be ~0 after centering, got {max_com_component:.2e}"
# confirm original eclipsed is unchanged
assert not np.allclose(eclipsed.coords, mol_copy.coords), \
    "eclipsed original should still be at its original position"
print(f"✓  Max COM component after centering: {max_com_component:.2e} Å")

---
## Section 5 — RMSD Without Alignment

The RMSD between two structures with the **same atom ordering** is:

$$\text{RMSD} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}|\vec{r}_i - \vec{r}_i^{\,\text{ref}}|^2}$$

In AaronTools:
```python
val = mol.RMSD(ref)   # no alignment — measures raw positional difference
```

Without alignment, even an identical molecule placed 5 Å away will report a large RMSD. This is generally not useful for comparing molecular shapes — but it is useful as a quick sanity check on whether two files truly represent the same geometry.

In [ ]:
# ── EXAMPLE — raw RMSD (no alignment) ────────────────────────────────────
ref   = Geometry("ethane_staggered.xyz")
ecl   = Geometry("ethane_eclipsed.xyz")
disp  = Geometry("ethane_displaced.xyz")   # same geometry, different position

rmsd_ecl  = ref.RMSD(ecl)
rmsd_disp = ref.RMSD(disp)

print(f"RMSD(staggered, eclipsed) — no align : {rmsd_ecl:.4f} Å")
print(f"RMSD(staggered, displaced) — no align: {rmsd_disp:.4f} Å")
print()
print("The displaced structure is IDENTICAL to staggered but looks very different")
print("without alignment because it lives in a different part of space.")

#### ✏️ Practice 5.1
Compute the RMSD between a **copy of `staggered` that you rotate 30° around the z-axis** and the original `staggered`, without alignment. Store:
- `rotated` — a copy of `staggered` rotated 30° around `[0, 0, 1]`
- `rmsd_rotated` — the RMSD (no alignment) between `staggered` and `rotated`
- `rmsd_self` — the RMSD of `staggered` against itself (should be exactly 0)

In [ ]:
# SOLUTION — Practice 5.1
rotated      = staggered.copy()
rotated.rotate([0, 0, 1], 30)        # 30° around z-axis

rmsd_rotated = staggered.copy().RMSD(rotated)     # no alignment
rmsd_self    = float(staggered.RMSD(staggered.copy()))

print(f"Self-RMSD         : {rmsd_self:.2e} Å")
print(f"RMSD(30° rotation): {rmsd_rotated:.4f} Å  (inflated by rotation, no alignment)")


In [ ]:
# TEST 5.1
assert rmsd_self < 1e-10, f"Self-RMSD must be 0, got {rmsd_self}"
assert rmsd_rotated > 0.1, "Rotating 30° should produce a non-trivial RMSD without alignment"
print(f"✓  RMSD(self)              : {rmsd_self:.2e} Å")
print(f"   RMSD(30° rotated, no align): {rmsd_rotated:.4f} Å")
print("   (same molecule, but rotation inflates the RMSD without alignment)")

---
## Section 6 — RMSD With Alignment: The Kabsch Algorithm

To get a meaningful structural comparison, we need to find the rotation and translation that **minimise** the RMSD before measuring it. The standard algorithm for this is the **Kabsch algorithm** (1976), which uses Singular Value Decomposition (SVD) to find the optimal rotation matrix in closed form.

The steps are:
1. Translate both structures so their centers of mass coincide at the origin.
2. Compute the covariance matrix $H = P^T Q$ where $P$ and $Q$ are the coordinate matrices of the two structures.
3. Decompose $H = U \Sigma V^T$ via SVD.
4. The optimal rotation matrix is $R = V U^T$ (with a sign correction if $\det(R) = -1$).
5. Apply $R$ to one structure and compute the RMSD.

AaronTools does all of this internally when you pass `align=True`:

```python
rmsd_val = mol.RMSD(ref, align=True)
```

**`align=True` modifies `mol` in place** — after the call, `mol.coords` contains the optimally rotated and translated coordinates. The return value is the minimised RMSD.

In [ ]:
# ── EXAMPLE — RMSD with Kabsch alignment ─────────────────────────────────
ref  = Geometry("ethane_staggered.xyz")
ecl  = Geometry("ethane_eclipsed.xyz").copy()   # copy so we can align it
disp = Geometry("ethane_displaced.xyz").copy()

rmsd_ecl_before  = ref.RMSD(ecl)
rmsd_disp_before = ref.RMSD(disp)

# Now align and measure
rmsd_ecl_after  = ecl.RMSD(ref,  align=True)
rmsd_disp_after = disp.RMSD(ref, align=True)

print("                              Before align    After align")
print(f"  staggered vs eclipsed  :    {rmsd_ecl_before:8.4f} Å     {rmsd_ecl_after:8.4f} Å")
print(f"  staggered vs displaced :    {rmsd_disp_before:8.4f} Å     {rmsd_disp_after:8.4f} Å")
print()
print("After alignment the displaced structure (identical geometry, different")
print("position) gives RMSD ≈ 0.  The eclipsed conformer gives a non-zero RMSD")
print("because the H atoms are genuinely in different positions.")

#### ✏️ Practice 6.1
- Make a copy of `staggered` called `test`, rotate it 45° around `[1, 1, 0]`, then translate it by `[3, -2, 1]`.
- `rmsd_before` — RMSD between `staggered` and `test` **before** alignment
- `rmsd_after` — RMSD between `staggered` and `test` **after** alignment (call `test.RMSD(staggered, align=True)`)
- `improved` — a boolean: `True` if `rmsd_after < rmsd_before`

In [ ]:
# SOLUTION — Practice 6.1
test = staggered.copy()
test.rotate([1, 1, 0], 45)
test.coord_shift([3, -2, 1])

rmsd_before = staggered.RMSD(test)                        # no alignment
test2       = staggered.copy()
test2.rotate([1, 1, 0], 45)
test2.coord_shift([3, -2, 1])
rmsd_after  = float(test2.RMSD(staggered.copy(), align=True))   # with alignment
improved    = rmsd_after < rmsd_before

print(f"RMSD before alignment: {rmsd_before:.4f} Å")
print(f"RMSD after  alignment: {rmsd_after:.2e} Å")
print(f"Improved: {improved}")


In [ ]:
# TEST 6.1
assert rmsd_before > 1.0, "A 45° rotation + 3 Å translation should give a large unaligned RMSD"
assert rmsd_after < 1e-4, f"After alignment the RMSD should be ~0 (identical molecule), got {rmsd_after:.4f}"
assert improved is True, "improved should be True"
print(f"✓  RMSD before alignment: {rmsd_before:.4f} Å")
print(f"   RMSD after  alignment: {rmsd_after:.2e} Å (effectively 0)")

---
## Section 7 — Heavy-Atom RMSD

Hydrogen positions are often less reliable in computational chemistry:
- X-ray crystallography determines H positions poorly (X-rays scatter from electrons, and H has only one).
- Many force fields and semiempirical methods are less accurate for H than for heavy atoms.
- Freely rotating methyl groups can produce large RMSD values that do not reflect meaningful structural differences.

It is therefore common to report **heavy-atom RMSD** (all non-hydrogen atoms) alongside or instead of all-atom RMSD.

```python
rmsd_heavy = mol.RMSD(ref, align=True, heavy_only=True)
```

In [ ]:
# ── EXAMPLE — all-atom vs heavy-atom RMSD ────────────────────────────────
ref = Geometry("ethane_staggered.xyz")
ecl = Geometry("ethane_eclipsed.xyz").copy()

rmsd_all   = ecl.RMSD(ref, align=True)

# Reload eclipsed for a fresh unaligned copy for the heavy-only comparison
ecl2 = Geometry("ethane_eclipsed.xyz").copy()
rmsd_heavy = ecl2.RMSD(ref, align=True, heavy_only=True)

print(f"All-atom RMSD  (staggered vs eclipsed): {rmsd_all:.4f} Å")
print(f"Heavy-atom RMSD(staggered vs eclipsed): {rmsd_heavy:.4f} Å")
print()
print("For ethane the only heavy atoms are the two carbons, which don't move")
print("between conformers — so the heavy-atom RMSD is ~0 even though the")
print("hydrogen positions differ substantially.")

#### ✏️ Practice 7.1
Compute both RMSD values between `staggered` and `displaced`, then compare:
- `rmsd_all_disp` — all-atom RMSD with alignment
- `rmsd_heavy_disp` — heavy-atom RMSD with alignment
- `ratio` — `rmsd_all_disp / rmsd_heavy_disp` (will raise `ZeroDivisionError` if heavy RMSD = 0 — use `float('inf')` in that case)

Think about what you expect: the displaced structure is identical to staggered (just moved in space), so both should be ≈ 0 after alignment.

In [ ]:
# SOLUTION — Practice 7.1
ref_copy        = Geometry("ethane_staggered.xyz")
disp_all_copy   = Geometry("ethane_displaced.xyz").copy()
disp_heavy_copy = Geometry("ethane_displaced.xyz").copy()

rmsd_all_disp   = float(disp_all_copy.RMSD(ref_copy.copy(),   align=True))
rmsd_heavy_disp = float(disp_heavy_copy.RMSD(ref_copy.copy(), align=True, heavy_only=True))
ratio           = rmsd_all_disp / rmsd_heavy_disp if rmsd_heavy_disp > 1e-12 else float('inf')

print(f"All-atom RMSD   (displaced vs staggered): {rmsd_all_disp:.2e} Å")
print(f"Heavy-atom RMSD (displaced vs staggered): {rmsd_heavy_disp:.2e} Å")


In [ ]:
# TEST 7.1
assert rmsd_all_disp < 1e-3, f"Displaced is identical to staggered — all-atom RMSD should be ~0, got {rmsd_all_disp}"
assert rmsd_heavy_disp < 1e-3, f"Heavy-atom RMSD should also be ~0, got {rmsd_heavy_disp}"
print(f"✓  All-atom RMSD  : {rmsd_all_disp:.2e} Å")
print(f"   Heavy-atom RMSD: {rmsd_heavy_disp:.2e} Å")
print("   Both ~0: alignment correctly identifies these as the same structure.")

---
## Section 8 — Targeted Alignment: Aligning on a Fragment

Sometimes you want to align only on a **subset** of atoms — for example, aligning two molecules on their shared scaffold while letting the rest of the structure float. AaronTools supports this via the `targets` and `ref_targets` parameters:

```python
rmsd = mol.RMSD(
    ref,
    align=True,
    targets="1,2",      # atom indices (1-based) in mol used for fitting
    ref_targets="1,2",  # corresponding atom indices in ref
)
```

You can also specify elements: `targets="C"` selects all carbon atoms. After the call, the **whole molecule** has been rotated/translated to minimise the RMSD over the target atoms — but the RMSD value returned is computed over all atoms unless you also restrict with `heavy_only`.

A common use case: aligning two structures on just the carbon skeleton, then examining where the functional groups end up.

In [ ]:
# ── EXAMPLE — align on carbon atoms only ─────────────────────────────────
ref = Geometry("ethane_staggered.xyz")
ecl = Geometry("ethane_eclipsed.xyz").copy()

# Align eclipsed to staggered using ONLY the two carbon atoms (indices 1 and 2, 1-based)
rmsd_c_targeted = ecl.RMSD(
    ref,
    align=True,
    targets="C",      # use carbon atoms in ecl for the fit
    ref_targets="C"   # and the corresponding carbons in ref
)

print(f"RMSD after carbon-targeted alignment: {rmsd_c_targeted:.4f} Å")
print("(This aligns the C-C axis; H atoms are placed wherever the rotation puts them)")

#### ✏️ Practice 8.1
Align `eclipsed` to `staggered` in two ways and compare:
- `rmsd_full_align` — standard all-atom alignment (`targets` not specified)
- `rmsd_carbon_align` — alignment using only `targets="C"` and `ref_targets="C"`
- `carbon_aligns_better` — a boolean: `True` if `rmsd_carbon_align < rmsd_full_align`

Think about which should be smaller and why.

In [ ]:
# SOLUTION — Practice 8.1
ref  = Geometry("ethane_staggered.xyz")
ecl1 = Geometry("ethane_eclipsed.xyz").copy()
ecl2 = Geometry("ethane_eclipsed.xyz").copy()

rmsd_full_align   = float(ecl1.RMSD(ref.copy(), align=True))
rmsd_carbon_align = float(ecl2.RMSD(ref.copy(), align=True, targets="C", ref_targets="C"))
carbon_aligns_better = rmsd_carbon_align < rmsd_full_align

print(f"Full alignment RMSD  : {rmsd_full_align:.4f} Å")
print(f"Carbon-targeted RMSD : {rmsd_carbon_align:.4f} Å")
print(f"Carbon aligns better : {carbon_aligns_better}")


In [ ]:
# TEST 8.1
assert rmsd_full_align > 0, "Full alignment RMSD should be > 0 for conformers"
assert rmsd_carbon_align >= 0, "Carbon-targeted RMSD should be >= 0"
print(f"✓  Full alignment RMSD  : {rmsd_full_align:.4f} Å")
print(f"   Carbon-targeted RMSD : {rmsd_carbon_align:.4f} Å")
print(f"   Carbon aligns better : {carbon_aligns_better}")
print()
print("Carbon-targeted alignment minimizes the C-C distance only, so it can")
print("achieve a lower C-based RMSD at the cost of H atoms being further apart.")

---
## Section 9 — RMSD_permute: Handling Symmetric Molecules

Consider a molecule with several identical atoms — ammonia (NH₃), benzene (C₆H₆), or even just two ends of ethane. Depending on how the atoms happen to be numbered in two different files, a standard `RMSD` call may pair H atom 1 in structure A with H atom 2 in structure B and report a misleadingly large RMSD for what are actually identical structures.

`RMSD_permute` fixes this by trying **all valid permutations** of same-element atoms and returning the minimum RMSD found:

```python
rmsd = mol.RMSD_permute(ref, align=True)
```

This is more expensive than a single `RMSD` call — the cost grows factorially with the number of identical atoms — but for small molecules like peptide fragments or simple organics it is typically fast.

In [ ]:
# ── EXAMPLE — RMSD_permute on a relabelled molecule ───────────────────────
# Create a version of staggered ethane where the H atoms on C1 are renumbered
# (swap H3 and H4) — standard RMSD will see a 'difference' that is not real.

import copy as py_copy

ref  = Geometry("ethane_staggered.xyz")
perm = Geometry("ethane_staggered.xyz").copy()

# Manually swap the coords of H atoms at index 2 and 3 (0-based)
coords_2 = perm.atoms[2].coords.copy()
perm.atoms[2].coords = perm.atoms[3].coords.copy()
perm.atoms[3].coords = coords_2

rmsd_std  = perm.copy().RMSD(ref,         align=True)
rmsd_perm = perm.copy().RMSD_permute(ref, align=True)

print(f"Standard RMSD after swapping two H labels: {rmsd_std:.4f} Å")
print(f"RMSD_permute after swapping two H labels : {rmsd_perm:.4f} Å")
print()
print("RMSD_permute correctly identifies these as the same structure (RMSD ≈ 0)")
print("by trying the permutation that maps H3→H4 and H4→H3.")

#### ✏️ Practice 9.1
Create `perm2` by copying `staggered` and swapping **all three H atoms on C2** (indices 5, 6, 7) by cycling them: position 5←6, 6←7, 7←5. Then compute:
- `rmsd_standard` — standard `RMSD` with alignment
- `rmsd_permuted` — `RMSD_permute` with alignment
- `permute_lower` — boolean: `True` if `rmsd_permuted < rmsd_standard`

In [ ]:
# SOLUTION — Practice 9.1
ref   = Geometry("ethane_staggered.xyz")
perm2 = Geometry("ethane_staggered.xyz").copy()

# Cycle coords: 5←6, 6←7, 7←5 (0-based indices)
c5 = perm2.atoms[5].coords.copy()
c6 = perm2.atoms[6].coords.copy()
c7 = perm2.atoms[7].coords.copy()
perm2.atoms[5].coords = c6
perm2.atoms[6].coords = c7
perm2.atoms[7].coords = c5

rmsd_standard = float(perm2.copy().RMSD(ref.copy(),          align=True))
rmsd_permuted = float(perm2.copy().RMSD_permute(ref.copy(),  align=True))
permute_lower = rmsd_permuted < rmsd_standard

print(f"Standard RMSD : {rmsd_standard:.4f} Å")
print(f"RMSD_permute  : {rmsd_permuted:.2e} Å")
print(f"Permute lower : {permute_lower}")


In [ ]:
# TEST 9.1
assert rmsd_standard > 1e-4, "Standard RMSD should be non-zero after permuting H labels"
assert rmsd_permuted < 1e-3, f"RMSD_permute should find the match and return ~0, got {rmsd_permuted:.4f}"
assert permute_lower is True
print(f"✓  Standard RMSD : {rmsd_standard:.4f} Å")
print(f"   RMSD_permute  : {rmsd_permuted:.2e} Å (correctly identifies identical structure)")

---
## Section 10 — Filtering a Conformer Ensemble

Conformer search algorithms (and grid scans of torsion angles) produce many structures, often with redundant near-duplicates. A standard post-processing step is to cluster structures by RMSD and keep only one representative from each cluster — the one with the lowest energy.

The simplest version of this is **greedy RMSD filtering**: iterate through the list of conformers in energy order; if a conformer has RMSD > threshold from all previously accepted structures, accept it; otherwise discard it.

```python
THRESHOLD = 0.25   # Å — structures closer than this are considered duplicates
```

The cell below creates a small synthetic conformer set to demonstrate the approach.

In [ ]:
# ── EXAMPLE — greedy conformer filtering ────────────────────────────────
import os, textwrap

# Build a synthetic ensemble: copies of staggered and eclipsed with small noise added
rng = np.random.default_rng(42)
conformer_files = []
template_s = Geometry("ethane_staggered.xyz")
template_e = Geometry("ethane_eclipsed.xyz")

for i in range(6):
    base = template_s if i < 3 else template_e
    conf = base.copy()
    # add tiny noise (< 0.05 Å) so they are near-duplicates, not exact copies
    for atom in conf.atoms:
        atom.coords += rng.normal(0, 0.02, 3)
    fname = f"conformer_{i}.xyz"
    conf.write(fname)
    conformer_files.append(fname)

# Greedy filtering
THRESHOLD = 0.25   # Å
accepted = []

for fname in conformer_files:
    candidate = Geometry(fname)
    is_duplicate = False
    for ref_geom in accepted:
        test_copy = candidate.copy()
        r = test_copy.RMSD(ref_geom.copy(), align=True)
        if r < THRESHOLD:
            is_duplicate = True
            break
    if not is_duplicate:
        accepted.append(candidate)
        print(f"  ACCEPTED: {fname}")
    else:
        print(f"  rejected: {fname} (duplicate)")

print(f"\n{len(accepted)} unique structures kept from {len(conformer_files)} candidates.")

#### ✏️ Practice 10.1
Implement the same greedy filter but wrap it in a reusable function and apply it with **two different thresholds** to see how the threshold affects the result.

- `filter_conformers(filenames, threshold)` — returns a list of accepted `Geometry` objects
- `n_kept_tight` — number kept with `threshold = 0.10` Å (tight)
- `n_kept_loose` — number kept with `threshold = 0.50` Å (loose)
- `tight_keeps_more` — boolean: `True` if the tight threshold keeps more structures than the loose one

In [ ]:
# SOLUTION — Practice 10.1
def filter_conformers(filenames, threshold):
    """
    Greedy RMSD filter. Returns a list of accepted Geometry objects.
    A candidate is accepted if its aligned RMSD to every already-accepted
    structure exceeds threshold.
    """
    accepted = []
    for fname in filenames:
        candidate = Geometry(fname)
        is_duplicate = False
        for ref_geom in accepted:
            test_copy = candidate.copy()
            r = float(test_copy.RMSD(ref_geom.copy(), align=True))
            if r < threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            accepted.append(candidate)
    return accepted

n_kept_tight     = len(filter_conformers(conformer_files, 0.10))
n_kept_loose     = len(filter_conformers(conformer_files, 0.50))
tight_keeps_more = n_kept_tight > n_kept_loose

print(f"Tight (0.10 Å): {n_kept_tight} structures kept")
print(f"Loose (0.50 Å): {n_kept_loose} structures kept")


In [ ]:
# TEST 10.1
assert callable(filter_conformers), "filter_conformers should be a function"
assert isinstance(n_kept_tight, int) and n_kept_tight > 0
assert isinstance(n_kept_loose, int) and n_kept_loose > 0
assert n_kept_tight >= n_kept_loose, "Tight threshold should keep >= as many structures as loose"
assert tight_keeps_more == (n_kept_tight > n_kept_loose)
print(f"✓  Tight (0.10 Å): {n_kept_tight} structures kept")
print(f"   Loose (0.50 Å): {n_kept_loose} structures kept")

---
## Section 11 — Connecting to Gaussian: Geometry from a `.log` File

In Parts 1 and 2 you wrote a custom Gaussian parser from scratch. AaronTools can do the same job in one line — and it will handle edge cases (multiple optimisation steps, failed jobs, frequency calculations) automatically.

For a completed geometry optimisation, `Geometry("job.log")` loads the **final Standard orientation** from the log file. This is the optimised structure.

Suppose you want to compare the geometry your custom parser extracted to what AaronTools reads — a useful cross-check. Or suppose you have a reference structure from the Cambridge Structural Database in `.xyz` format and want to know how well your DFT optimisation reproduced it.

In [ ]:
# ── EXAMPLE — load from Gaussian .log and inspect ────────────────────────
# Uses the water.log file from the Track 1 folder
import os
log_path = os.path.join("..", "water.log") if os.path.exists("../water.log") else "water.log"

water = Geometry(log_path)

print(f"Atoms loaded from water.log: {water.n_atoms}")
print(f"Formula: H={sum(1 for a in water.atoms if a.element=='H')} "
      f"O={sum(1 for a in water.atoms if a.element=='O')}")
print()
print("Atomic positions (Å):")
for atom in water.atoms:
    print(f"  {atom.element:2s}  {atom.coords[0]:9.5f}  {atom.coords[1]:9.5f}  {atom.coords[2]:9.5f}")

#### ✏️ Practice 11.1
Using the `water` geometry loaded from `water.log`:
- `o_atom` — the oxygen `Atom` object (hint: find the atom with `.element == "O"`)
- `h_atoms` — a list of the two hydrogen `Atom` objects
- `oh1_dist` — O−H bond length to the first H (Å)
- `oh2_dist` — O−H bond length to the second H (Å)
- `hoh_angle_deg` — the H−O−H bond angle in degrees using the dot-product formula from Part 2

In [ ]:
# SOLUTION — Practice 11.1
o_atom  = next(a for a in water.atoms if a.element == "O")
h_atoms = [a for a in water.atoms if a.element == "H"]

oh1_dist = float(np.linalg.norm(o_atom.coords - h_atoms[0].coords))
oh2_dist = float(np.linalg.norm(o_atom.coords - h_atoms[1].coords))

# Dot-product bond angle (same formula as Part 2)
v1 = h_atoms[0].coords - o_atom.coords
v2 = h_atoms[1].coords - o_atom.coords
cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
hoh_angle_deg = float(np.degrees(np.arccos(np.clip(cos_angle, -1, 1))))

print(f"O-H distances  : {oh1_dist:.4f} Å, {oh2_dist:.4f} Å")
print(f"H-O-H angle    : {hoh_angle_deg:.2f}°")


In [ ]:
# TEST 11.1
assert o_atom.element == "O", "o_atom should be the oxygen atom"
assert len(h_atoms) == 2, "h_atoms should have 2 hydrogen atoms"
assert all(a.element == "H" for a in h_atoms)
assert 0.90 < oh1_dist < 1.10, f"O-H distance should be ~0.96 Å, got {oh1_dist:.4f}"
assert 0.90 < oh2_dist < 1.10, f"O-H distance should be ~0.96 Å, got {oh2_dist:.4f}"
assert 100.0 < hoh_angle_deg < 115.0, f"H-O-H angle should be ~104.5°, got {hoh_angle_deg:.2f}°"
print(f"✓  O-H distances: {oh1_dist:.4f} Å, {oh2_dist:.4f} Å")
print(f"   H-O-H angle  : {hoh_angle_deg:.2f}°")
print("(Typical DFT values: O-H ≈ 0.961 Å, H-O-H ≈ 104.5°)")

---
## Summary

You now have a working toolkit for structure alignment with AaronTools:

- **Loading structures** with `Geometry(filename)` from `.xyz`, `.log`, `.com`, and other formats
- **Inspecting geometries** via `.atoms`, `.coords`, `.n_atoms`, and `Atom.element` / `Atom.coords`
- **Centring molecules** with `COM()` and `coord_shift()`
- **Measuring RMSD** with and without alignment via `RMSD(ref, align=False/True)`
- **Heavy-atom RMSD** with `heavy_only=True`
- **Targeted alignment** on a subset of atoms with `targets=` and `ref_targets=`
- **Permutation-aware RMSD** with `RMSD_permute()` for symmetric molecules
- **Conformer filtering** using greedy pairwise RMSD comparison
- **Reading Gaussian output** directly into a `Geometry` object

**Where to go next:**
- Use `Geometry.write("output.com")` to write AaronTools-manipulated structures back to Gaussian input format for further calculations.
- Explore `AaronTools.substituent` for swapping functional groups on a scaffold.
- Look at `AaronTools.component` for building transition-metal complexes.
- Combine with Part 2's parser: extract a series of geometries along an IRC and plot RMSD from the reactant as a function of the reaction coordinate.